# ZestXML benchmarks

Runs every method on **GZ-NPM** and **GZ-Reuters-90** and prints two comparison tables.
Each stage is independent — run stage 0, then whichever of 1–5 you want. The tables in
stage 6 include whatever has been run, so a partial run still gives a partial table.

| stage | cost on an A100 |
|---|---|
| 0 setup + datasets | ~3 min |
| 1 ZestXML and its variants | ~5 min |
| 2 classical baselines | ~5 min |
| 3 SPLADE | ~10 min |
| 4 pretrained encoders | ~10 min |
| 5 Renee | 45–90 min |

Everything but stage 5 also runs on CPU, more slowly. Stage 5 needs a GPU.

## 0 — setup

In [ ]:
!nvidia-smi -L || echo 'no GPU: stages 0-4 still work, stage 5 does not'

In [ ]:
%cd /content
!rm -rf zestxml
!git clone -q -b claude/pytorch-rewrite-fhuwl4 https://github.com/hanialshater/zestxml.git
%cd /content/zestxml
!pip install -q scikit-learn

# The GZ-NPM / GZ-Reuters builders take several minutes and this stage does not use them.
# Uncomment to restore them for the earlier stages.
# !pip install -q nltk && python -c "import nltk; nltk.download('reuters', quiet=True)"
# !python benchmarks/datasets/make_npm.py GZXML-Datasets/GZ-NPM
# !python benchmarks/datasets/make_reuters.py GZXML-Datasets/GZ-Reuters-90
print('ready')

## 1 — ZestXML and its variants

The reference configuration per dataset, then the two things that change how an *unseen*
label is reached: a fuzzy direct map, and label feature-bag expansion. Both need word
vectors; GloVe-100d is fetched below.

In [ ]:
# SKIPPED to keep the run short — already measured, see benchmarks/RESULTS.md.
# Uncomment this cell to re-run it.
# # GloVe-100d (gensim-data release asset, ~130 MB). Any word2vec/GloVe/fastText text
# # file works -- pass its path as VECTORS.
# VECTORS = '/content/glove100.gz'
# !wget -q --show-progress -O {VECTORS} \
#   https://github.com/RaRe-Technologies/gensim-data/releases/download/glove-wiki-gigaword-100/glove-wiki-gigaword-100.gz
# import os; print('vectors:', os.path.exists(VECTORS) and os.path.getsize(VECTORS))

In [ ]:
# SKIPPED to keep the run short — already measured, see benchmarks/RESULTS.md.
# Uncomment this cell to re-run it.
# import os, sys
# sys.path.insert(0, '/content/zestxml')
# from zestxml import ZestXML

# # the reference configuration for each dataset (Results/*2exact/params.txt)
# CFG = {
#     'GZ-NPM':        dict(shortyK=100, bs_count=40, bs_alpha=0.02, bs_direct_wt=0.8),
#     'GZ-Reuters-90': dict(shortyK=50,  bs_count=20, bs_alpha=0.02, bs_direct_wt=0.8),
# }
# COMMON = dict(bilinear_classifier_cost=5, bilinear_normalize=0, device='auto', num_thread=0)

# def run(dataset, tag, **extra):
#     m = ZestXML(f'GZXML-Datasets/{dataset}', f'Results/{tag}', **CFG[dataset], **COMMON, **extra)
#     m.fit(); m.predict()
#     print(f'--- {tag}'); m.evaluate(); print()

# for ds, short in (('GZ-NPM', 'npm'), ('GZ-Reuters-90', 'reu')):
#     run(ds, f'{short}-reference')
#     if os.path.exists(VECTORS):
#         run(ds, f'{short}-directmap', direct_map='vectors', direct_vectors=VECTORS,
#             direct_topk=3, direct_min_sim=0.5, direct_fallback=1)

In [ ]:
# SKIPPED to keep the run short — already measured, see benchmarks/RESULTS.md.
# Uncomment this cell to re-run it.
# # label feature-bag expansion rebuilds the dataset, so it goes through build_dataset
# if os.path.exists(VECTORS):
#     for ds in ('GZ-Reuters-90', 'GZ-NPM'):
#         cfg = ' '.join(f'{k}={v}' for k, v in CFG[ds].items())
#         !python benchmarks/expansion_check.py GZXML-Datasets/{ds} {VECTORS} {cfg} 2>&1 | grep -E 'expansion   :|delta'

## 2 — classical baselines

BM25, kNN and tf-idf centroid (one script, three outputs), and one-vs-all linear.
The point of these is the unseen column: everything except BM25 scores exactly zero there.

In [ ]:
# SKIPPED to keep the run short — already measured, see benchmarks/RESULTS.md.
# Uncomment this cell to re-run it.
# !python benchmarks/baselines/classical.py GZXML-Datasets/GZ-NPM classical 2>&1 | tail -2
# !python benchmarks/baselines/classical.py GZXML-Datasets/GZ-Reuters-90 classicalR 2>&1 | tail -2
# !python benchmarks/baselines/ova_linear.py -data GZXML-Datasets/GZ-NPM -res_dir Results/ova_linear -epochs 15 2>&1 | tail -2
# !python benchmarks/baselines/ova_linear.py -data GZXML-Datasets/GZ-Reuters-90 -res_dir Results/ova_linear_reuters -epochs 15 2>&1 | tail -2

## 3 — SPLADE-style learned sparse

Four ablations. `drop_label_id` removes the per-label feature, which is what makes the
difference between a method that can reach an unseen label and one that cannot.

In [ ]:
# SKIPPED to keep the run short — already measured, see benchmarks/RESULTS.md.
# Uncomment this cell to re-run it.
# for tag, flags in [('splade-base',  '-drop_label_id 0 -norm_labels 0'),
#                    ('splade-nodid', '-drop_label_id 1 -norm_labels 0'),
#                    ('splade-norm',  '-drop_label_id 0 -norm_labels 1'),
#                    ('splade-both',  '-drop_label_id 1 -norm_labels 1')]:
#     !python benchmarks/baselines/splade.py -data GZXML-Datasets/GZ-NPM -res Results/{tag} -epochs 6 {flags} 2>&1 | tail -2

## 4 — pretrained encoders

Two questions. Can a sentence encoder retrieve candidates the lexical shortlist misses
(the ceiling on npm is 71.8% recall, and only 54% of *unseen* positives get in)? And is
ZestXML better on a hybrid shortlist than on its own?

In [ ]:
# SKIPPED to keep the run short — already measured, see benchmarks/RESULTS.md.
# Uncomment this cell to re-run it.
# !pip install -q sentence-transformers
# !python benchmarks/hf_dense_probe.py --data GZXML-Datasets/GZ-NPM \
#     --lexical Results/npm-reference/shortlist.bin --out Results/npm-hybrid \
#     --model sentence-transformers/all-MiniLM-L6-v2 --k 100

In [ ]:
# SKIPPED to keep the run short — already measured, see benchmarks/RESULTS.md.
# Uncomment this cell to re-run it.
# # rescore with the hybrid candidates; stage 1 of the model is unchanged, so reuse it
# !cp -r Results/npm-reference/model Results/npm-hybrid/model 2>/dev/null || true
# m = ZestXML('GZXML-Datasets/GZ-NPM', 'Results/npm-hybrid', **CFG['GZ-NPM'], **COMMON,
#             shortlist_file='Results/npm-hybrid/hybrid_shortlist.bin')
# m.predict(); m.evaluate()

## 5 — Renee (Microsoft, MLSys 2023)

End-to-end one-vs-all over a transformer encoder, no shortlisting. Needs a GPU and
45–90 min. Three things had to be patched to make it run on a current Colab: `apex` is
not on PyPI (`pip install apex` fetches an unrelated Pyramid library), `batch_encode_plus`
was removed from `transformers`, and the tokenised row counts have to be checked because
Renee maps line N of `trn_X.txt` to row N of `trn_X_Y.txt` without ever verifying it.

In [ ]:
# SKIPPED to keep the run short — already measured, see benchmarks/RESULTS.md.
# Uncomment this cell to re-run it.
# %%bash
# set -e
# cd /content
# [ -d renee/.git ] || git clone -q https://github.com/microsoft/renee.git
# pip install -q transformers cython seaborn
# pip install -q git+https://github.com/kunaldahiya/pyxclib.git
# pip uninstall -y -q apex 2>/dev/null; true

# cd /content/renee
# cp /content/zestxml/benchmarks/colab/apex.py apex.py     # torch stand-in for the fused optimizers
# sed -i 's/tokenizer\.batch_encode_plus(/tokenizer(/' utils/CreateTokenizedFiles.py
# mkdir -p xc/Datasets && rm -rf xc/Datasets/GZ-NPM xc/Datasets/GZ-NPM-Aug
# cp -r /content/zestxml/GZXML-Datasets/GZ-NPM xc/Datasets/GZ-NPM
# python -c "import apex; print('apex shim ok:', apex.optimizers.FusedAdam)"

In [ ]:
# SKIPPED to keep the run short — already measured, see benchmarks/RESULTS.md.
# Uncomment this cell to re-run it.
# %cd /content/renee
# !python -W ignore -u utils/CreateTokenizedFiles.py --data-dir xc/Datasets/GZ-NPM \
#   --max-length 32 --tokenizer-type bert-base-uncased --tokenize-label-texts
# !python utils/CreateAugData.py --data-dir xc/Datasets/GZ-NPM \
#   --tokenization-folder bert-base-uncased-32 --max-len 32

# import os
# d = 'xc/Datasets/GZ-NPM-Aug/bert-base-uncased-32'
# for f in sorted(os.listdir(d)):
#     print(f'{f:34s} {os.path.getsize(f"{d}/{f}") // (8 * 32):>7d} rows')
# print('expect trn_doc 28350 (25127 points + 3223 label texts), tst_doc 8376, lbl 3223')

In [ ]:
# SKIPPED to keep the run short — already measured, see benchmarks/RESULTS.md.
# Uncomment this cell to re-run it.
# # drop --epochs to 20 for a cheaper first look
# !cd /content/renee && python main.py --epochs 50 --batch-size 32 --lr1 0.05 --lr2 1e-5 \
#   --warmup 1000 --data-dir xc/Datasets/GZ-NPM-Aug --maxlen 32 \
#   --tf sentence-transformers/msmarco-distilbert-base-v4 \
#   --dropout 0.85 --pre-tok --wd1 1e-4 --noloss --fp16xfc --expname gznpm-aug

In [ ]:
# SKIPPED to keep the run short — already measured, see benchmarks/RESULTS.md.
# Uncomment this cell to re-run it.
# # Renee's output layout is version dependent: find its matrix, then convert it so the
# # same evaluator scores both systems.
# import glob, scipy.sparse as sp, torch, sys
# sys.path.insert(0, '/content/zestxml')
# from zestxml.csr import CSR
# from zestxml.io import write_bin_smat, ensure_dir

# found = sorted(glob.glob('/content/renee/**/*.npz', recursive=True), key=os.path.getmtime)
# print('candidates:', *found, sep='\n  ')
# SCORES = found[-1] if found else ''   # or paste a path here

# if SCORES:
#     m = sp.load_npz(SCORES).tocsr()
#     print('renee scores', m.shape, m.nnz)
#     ensure_dir('/content/zestxml/Results/renee')
#     write_bin_smat(CSR(torch.as_tensor(m.indptr).long(), torch.as_tensor(m.indices).long(),
#                        torch.as_tensor(m.data).float(), m.shape),
#                    '/content/zestxml/Results/renee/score_mat.bin')

## 6 — the tables

Every row is recomputed from the score matrix on disk, so nothing here can drift from
the artifacts. `--scan` picks up any run not in the script's label list, including
whatever the stages above happened to produce.

In [ ]:
# SKIPPED to keep the run short — already measured, see benchmarks/RESULTS.md.
# Uncomment this cell to re-run it.
# %cd /content/zestxml
# !python benchmarks/results_table.py GZ-NPM --scan --md
# !python benchmarks/results_table.py GZ-Reuters-90 --scan --md

### Reading the tables

* Sort order is **unseen-label P@1**, which is what these datasets exist to measure.
* An unseen P@1 of **2.49** on npm (2.07 on Reuters) is the evaluator's tie-break floor,
  not a score: those matrices have no non-zero entry in any unseen column. Read it as 0.
  Every per-label classifier — OVA, kNN, centroid — lands there.
* Aggregate P@1 barely separates the ZestXML variants (all within ~0.1 on npm). The
  unseen column is where they differ, by up to 8 points.
* Training is only bit-reproducible at `num_thread=1`; above it, two identical runs differ
  by up to ~0.011 in the metrics. Treat differences of that size as noise.

## 7 — RQ-KMeans semantic tokens, with a real encoder

Residual k-means over embeddings, emitted as features: a document gets `rq0_<c> ...`, a
label gets `1_rq0_<c> ...`, and the direct map links them by string equality with no model
change. Measured on CPU with **GloVe means** this was neutral at best (see
`benchmarks/RESULTS.md`), but that encoder had a defect worth removing before believing the
result: documents were averaged over 200 tokens while label names were averaged over 1–2,
so the two sides sat in different regions of the space, and label names were then quantized
against *document*-cluster centroids.

Two encoders, answering different questions:

* **`all-MiniLM-L6-v2`** — was GloVe the limitation? One encoder for both sides, no
  out-of-vocabulary drops, 256-token window.
* **`clip-ViT-B-32`** — the shared image–text space. On a text corpus it is the weaker
  choice (77-token window truncates documents hard), but it is the *only* configuration
  that extends to images, because an image and a label **name** can be quantized with one
  codebook. Run it here to see what the shared space costs on text before relying on it.

`--block_split` is the variable that decides everything: on CPU, unseen P@1 fell 61.3 →
60.9 → 58.1 → 47.0 → 24.3 as the rq block took 0 → 10 → 25 → 50 → 100% of row mass. Sweep
low.

In [ ]:
# SKIPPED to keep the run short — already measured, see benchmarks/RESULTS.md.
# Uncomment this cell to re-run it.
# !pip install -q sentence-transformers
# import os, itertools, sys
# sys.path.insert(0, '/content/zestxml')
# os.chdir('/content/zestxml')

# DS = 'GZ-Reuters-90'                       # swap for GZ-NPM (slower: 25k documents)
# GRID = [('all-MiniLM-L6-v2', 0.10), ('all-MiniLM-L6-v2', 0.25),
#         ('clip-ViT-B-32',    0.10), ('clip-ViT-B-32',    0.25)]

# for model, split in GRID:
#     tag = f"Rq-{model.split('/')[-1]}-{split}"
#     !python benchmarks/rq_concat.py GZXML-Datasets/{DS} GZXML-Datasets/{tag} \
#         --model {model} -L 4 -K 64 --block_split {split} 2>&1 | grep -E 'embedded|labels: level|wrote'

In [ ]:
# SKIPPED to keep the run short — already measured, see benchmarks/RESULTS.md.
# Uncomment this cell to re-run it.
# from zestxml import ZestXML
# from zestxml.io import read_bin_smat, read_text_smat
# import torch

# CFG = {'GZ-Reuters-90': dict(shortyK=50, bs_count=20), 'GZ-NPM': dict(shortyK=100, bs_count=40)}[DS]
# COMMON = dict(bs_alpha=0.02, bs_direct_wt=0.8, bilinear_classifier_cost=5,
#               bilinear_normalize=0, device='auto', num_thread=0)

# truth = read_text_smat(f'GZXML-Datasets/{DS}/tst_X_Y.txt')
# uns = torch.zeros(truth.ncols, dtype=torch.bool)
# for l in open(f'GZXML-Datasets/{DS}/unseen_labels.txt'):
#     if l.strip(): uns[int(l.split()[0])] = True
# t = truth.to_dense() > 0; tu = t & uns[None, :]

# def run(ds, res):
#     m = ZestXML(ds, res, **CFG, **COMMON); m.fit(); m.predict()
#     sl = read_bin_smat(f'{res}/shortlist.bin').to_dense() != 0
#     return m.evaluate(verbose=False), 100.0*(sl & tu).sum().item()/tu.sum().item()

# rows = [('control (lexical)',) + run(f'GZXML-Datasets/{DS}', 'Results/Rq-ctrl')]
# for model, split in GRID:
#     tag = f"Rq-{model.split('/')[-1]}-{split}"
#     rows.append((f'{model.split("/")[-1]} @ {int(split*100)}%',) + run(f'GZXML-Datasets/{tag}', f'Results/{tag}'))

# print(f'\n{DS}')
# print('%-28s %7s %7s %9s %9s %9s' % ('arm','P@1','PSP@5','unseen','seen','unseen-rec'))
# for name, m, ur in rows:
#     print('%-28s %7.2f %7.2f %9.2f %9.2f %9.2f' % (
#         name, m['all labels']['P@1'], m['all labels']['PSP@5'],
#         m['unseen only']['P@1'], m['seen only']['P@1'], ur))

**What to read.** The CPU/GloVe baseline to beat, on GZ-Reuters-90: control P@1 86.35,
unseen P@1 61.28, seen 95.04, PSP@5 62.94, unseen shortlist recall 70.93; the best rq arm
reached unseen recall 94.68 while never beating control on unseen P@1 by more than ~1 point,
and a verifier found the overall P@1 gain to be seed noise.

So the bar is: **does a real encoder produce an unseen P@1 gain larger than ~1 point?** If
not, the CPU conclusion stands and the limitation was never the encoder — it was that
99–100% of unseen-label tokens are already in the document vocabulary, so an exact lexical
link already fires and semantic codes can only add a fuzzier version of a link already
present. Watch `unseen`, not `unseen-rec`: recall gains did not convert on CPU, and feeding
the unchanged lexical model 12.5 points more unseen recall moved unseen P@1 *down* 0.75.

**For an image corpus.** Nothing above uses images, because these datasets have none. The
path is `zestxml.quantize.encode_images(paths, 'clip-ViT-B-32')` to fit the codebook on
images, then `encode_texts(label_names, 'clip-ViT-B-32')` quantized through the **same**
codebook. Using different models for the two sides makes the codes a seen-label-only
feature — no unseen label could ever carry one — which defeats the purpose.

## 8 — label expansion with a modern encoder

Stage 7 compared GloVe, MiniLM and CLIP **inside RQ-KMeans**, where all three do nothing —
so it says little about the encoders and a lot about quantization. This stage puts them in
the mechanism that actually works.

Label expansion gives each label its nearest vocabulary neighbours as extra `1_<token>`
features, *alongside* its own tokens rather than replacing them. On GloVe it is the largest
gain measured anywhere in this repo: **Reuters unseen P@1 61.28 → 69.36 (+8.1)**. It is also
the one that fails hardest when the vector space does not know the vocabulary: **npm 52.11 →
44.88 (−7.2)**, because GloVe has never seen `webpack` or `eslint`.

That makes npm the interesting run. If a modern encoder is genuinely better here, it should
turn npm's −7.2 around; matching GloVe on Reuters only shows it is not worse.

**`min_sim` does not transfer between encoders.** Transformer spaces are anisotropic —
cosines between unrelated items sit far above zero — so 0.7, which is selective for GloVe,
may admit almost everything. Judge a setting by the `expansion :` line each build prints,
not by the threshold: what decided the sign in every measured run is **how many labels each
added feature lands on** — 1.25 gained 8 points, 1.54 with a tail to 19 lost 7.

In [ ]:
# SKIPPED to keep the run short — already measured, see benchmarks/RESULTS.md.
# Uncomment this cell to re-run it.
# !pip install -q sentence-transformers
# %cd /content/zestxml
# import os
# VECTORS = '/content/glove100.gz'      # from stage 1; only needed for the GloVe row

# # (encoder, k, cosine floor). Sweep the floor for the transformer rows -- 0.7 is a GloVe
# # number and means something different in a transformer space.
# ARMS = [
#     (VECTORS,            2, 0.7),     # the recorded GloVe baseline, for comparison
#     ('all-MiniLM-L6-v2', 2, 0.5),
#     ('all-MiniLM-L6-v2', 2, 0.7),
#     ('clip-ViT-B-32',    2, 0.7),
# ]
# DS   = 'GZ-Reuters-90'                # then rerun this cell with 'GZ-NPM'
# CFG  = {'GZ-Reuters-90': '', 'GZ-NPM': 'shortyK=100 bs_count=40'}[DS]

# for enc, k, floor in ARMS:
#     print('=' * 78, f'\n{enc}  k={k} floor={floor}\n')
#     !python benchmarks/expansion_check.py GZXML-Datasets/{DS} {enc} {k} {floor} {CFG} \
#         2>&1 | grep -E 'expander:|expansion   :|delta|labels have a token'

**Reading it.** Two lines matter per arm:

* `expansion : N added features, X labels each on average (max M)` — the predictor. Keep
  X near 1.3. If X is large or `max` runs into the tens, the added features cannot
  discriminate and the arm will lose however good the encoder is.
* `unseen only P@1 ... delta` — the payoff. GloVe's numbers to beat: **+8.08 on Reuters,
  −7.23 on npm**.

`seen only P@1` should stay near zero; expansion that buys unseen accuracy by giving up seen
accuracy is a trade, not a win, and should be reported as one.

If a transformer encoder fixes npm, that is the first result in this repo to break the
coverage rule — worth a second seed before believing it, since the control's own seed spread
on unseen P@1 is 0.38.

# (b) retrain on semantic candidates -- four arms, all measured here.
# The two shortlist files have different shapes (train vs test rows), so both are needed;
# passing only the test one would train on lexical negatives and rank semantic ones.
!python benchmarks/rq_retrain.py GZXML-Datasets/{DS} GZXML-Datasets/Multi-{DS}-m3 {CFG}

In [ ]:
# SKIPPED to keep the run short — already measured, see benchmarks/RESULTS.md.
# Uncomment this cell to re-run it.
# %cd /content/zestxml
# # npm is the informative one: 3223 labels and 77% GloVe token coverage, against Reuters'
# # 90 labels at ~100%, where lexical matching already has almost nothing left to gain.
# DS  = 'GZ-NPM'                                  # or 'GZ-Reuters-90'
# CFG = {'GZ-Reuters-90': 'shortyK=50 bs_count=20',
#        'GZ-NPM':        'shortyK=100 bs_count=40'}[DS]     # derived, so it cannot drift from DS
# ENC = '--model all-MiniLM-L6-v2'                # or: f'--vectors {VECTORS}'
# print(DS, CFG)

# # (a) multiple codes per label
# for m in (1, 3):
#     print('=' * 78, f'\nlabel_codes = {m}\n')
#     !python benchmarks/rq_concat.py GZXML-Datasets/{DS} GZXML-Datasets/Multi-{DS}-m{m} \
#         {ENC} -L 4 -K 64 --block_split 0.10 --label_codes {m} \
#         2>&1 | grep -E 'embedded|code-sets|labels: level 0|wrote'

In [ ]:
# SKIPPED to keep the run short — already measured, see benchmarks/RESULTS.md.
# Uncomment this cell to re-run it.
# # (b) retrain on semantic candidates -- four arms, all measured here.
# # The two shortlist files have different shapes (train vs test rows), so both are needed;
# # passing only the test one would train on lexical negatives and rank semantic ones.
# !python benchmarks/rq_retrain.py GZXML-Datasets/{DS} GZXML-Datasets/Multi-{DS}-m3 {CFG}

In [ ]:
# SKIPPED to keep the run short — already measured, see benchmarks/RESULTS.md.
# Uncomment this cell to re-run it.
# # (c) semantic pruning of the mined pattern. min_sim 0 is the control.
# # On Reuters this traded unseen accuracy for seen accuracy and did not pay for itself; the
# # argument for it is pattern-budget efficiency, which only bites on a large label set.
# from zestxml import ZestXML
# base = dict(bs_alpha=0.02, bs_direct_wt=0.8, bilinear_classifier_cost=5,
#             bilinear_normalize=0, num_thread=0,
#             **{k: int(v) for k, v in (o.split('=') for o in CFG.split())})

# print('%-12s %7s %7s %9s %9s' % ('min_sim', 'P@1', 'PSP@5', 'unseen', 'seen'))
# for s in (0.0, 0.15, 0.30):
#     m = ZestXML(f'GZXML-Datasets/{DS}', f'Results/Prune-{DS}-{s}', **base,
#                 **({} if s == 0 else dict(prune_vectors=VECTORS, prune_min_sim=s)))
#     m.fit(); m.predict(); r = m.evaluate(verbose=False)
#     print('%-12s %7.2f %7.2f %9.2f %9.2f' % (s, r['all labels']['P@1'],
#           r['all labels']['PSP@5'], r['unseen only']['P@1'], r['seen only']['P@1']))

**What each answers.**

* (a) watch `code-sets`: at `m=3` labels should be near-uniquely identified while
  `labels: level 0` stays unchanged, since rank-1 assignment is untouched. If unseen P@1
  still does not move, code granularity was never the limitation.
* (b) the row to read is **`lexical + union candidates`** against **`lexical control`**. Same
  features, same everything, only the candidate set differs — and now consistently between
  training and test. If this is still flat, "retrieval is not the constraint" is established
  rather than suspected.
* (c) `min_sim 0.0` is the control. Pruning that *helps* would be the first evidence that
  co-occurrence mining wastes pattern budget; pruning that costs unseen accuracy means
  co-occurrence was already picking the right pairs.

GloVe numbers to beat on Reuters: P@1 86.35, unseen 61.28, seen 95.04, PSP@5 62.94. Treat
anything under ~1 point of unseen P@1 as noise — the control's own seed spread is 0.38.

## 10 — EMMETT / IRENE: synthesize the missing classifier

[Yadav et al., KDD '24](https://dl.acm.org/doi/10.1145/3637528.3672046), from the ZestXML
authors. It inverts everything above. Each earlier stage tried to *describe* an unseen label
better — its tokens, its embedding, its semantic id. IRENE builds the classifier the label
does not have, out of the classifiers of labels it resembles:

    unseen "trail running" -> nearest SEEN labels: hiking, marathon, outdoor gear
                           -> a small attention layer combines their weight vectors
                           -> a classifier for "trail running"

Four arms. `one-vs-all` should sit at the evaluator's tie-break floor on the unseen split —
it has no classifier there at all, which is the whole problem. `+ mean synthesis` is the
no-learning ablation and must be beaten before the generator can be credited. Classifiers
live in the encoder's embedding space, which is what makes synthesis cheap.

In [ ]:
# SKIPPED — see benchmarks/RESULTS.md for what it measured.
# Uncomment this cell to re-run it.
# %cd /content/zestxml
# !pip install -q sentence-transformers

# # self-contained: nothing above this cell needs to have been run except the clone.
# DS = 'GZ-NPM'                     # 2937 seen labels to train the generator on; or 'GZ-Reuters-90'
# !python benchmarks/irene.py GZXML-Datasets/{DS} --model all-MiniLM-L6-v2 \
#     -k 16 --epochs 15 --gen_epochs 60

**What I already measured on CPU with GloVe means, and why it settles nothing.**
On GZ-Reuters-90: dual encoder P@1 3.91, one-vs-all 59.16, unseen P@1 2.07 → 8.08 (mean) →
6.77 (generator).

The mechanism works — synthesis lifts unseen labels off the 2.07 tie-break floor, which is
the thing one-vs-all fundamentally cannot do. But the *whole system* sits far below ZestXML
(86.35 P@1, 61.28 unseen), because a mean of GloVe vectors is a poor document
representation and every arm inherits that ceiling. The generator also lost to the plain
mean, which is unsurprising with 75 seen labels to train on.

So the numbers to watch here are **relative**: does `+ IRENE generator` beat `+ mean
synthesis`, and does `one-vs-all` plus either of them get within reach of ZestXML's unseen
P@1 (52.11 on npm)? A strong encoder is what the paper assumes, and npm's 2937 seen labels
are what the generator needs. If IRENE still loses to the mean on npm with MiniLM, the
learned generator is not earning its complexity at this scale — the paper's datasets are
270K–1.3M labels, two orders of magnitude larger.

## 11 — reproduce on the paper's own dataset

Everything above runs on GZ-Reuters-90 and GZ-NPM, which are **built by this repo**. They
have the right shape but no published numbers, so nothing measured on them can be checked
against anyone else's work. This stage runs **GZ-EURLex-4.3K** — the smallest dataset in the
ZestXML paper itself (45,000 train / 6,000 test, 4,108 seen + 163 unseen labels) — with the
hyper-parameters from the authors' own `run_eurlex.sh`.

Two things get checked, and the second is the one that matters:

1. **Does the port land where the paper says ZestXML lands?** Compare the `all labels` and
   `unseen only` rows against Table 2 of
   [Gupta et al., KDD '21](https://dl.acm.org/doi/pdf/10.1145/3447548.3467426). This
   validates the implementation against an external reference for the first time.
2. **Does the one positive result in this repo survive on a standard benchmark?** Semantic
   pattern pruning gave npm P@1 73.02 → 74.57. If that is a real effect it should appear
   here too; if it is an artifact of a self-built dataset, it will not.

In [ ]:
# SKIPPED — see benchmarks/RESULTS.md for what it measured.
# Uncomment this cell to re-run it.
# %cd /content/zestxml
# !pip install -q gdown
# import os
# if not os.path.exists('GZXML-Datasets/GZ-Eurlex-4.3K/trn_X_Xf.txt'):
#     !mkdir -p GZXML-Datasets && cd GZXML-Datasets && \
#         gdown -q "https://drive.google.com/uc?id=1j27bQZol6gOQ7AATawShcF4jXJr3Venb" && \
#         tar -xzf GZ-Eurlex-4.3K.tar.gz && rm -f GZ-Eurlex-4.3K.tar.gz
# !ls GZXML-Datasets/GZ-Eurlex-4.3K | head -20

# # a failing !command does not stop a notebook; check before spending a run on it
# NEED = ['trn_X_Xf.txt','tst_X_Xf.txt','Y_Yf.txt','trn_X_Y.txt','tst_X_Y.txt','Xf.txt','Yf.txt']
# gone = [f for f in NEED if not os.path.exists(f'GZXML-Datasets/GZ-Eurlex-4.3K/{f}')]
# assert not gone, f'download/extract failed, missing {gone}'
# print('GZ-Eurlex-4.3K ready')

In [ ]:
# SKIPPED — see benchmarks/RESULTS.md for what it measured.
# Uncomment this cell to re-run it.
# # the authors' own hyper-parameters, from run_eurlex.sh in the ZestXML repo
# from zestxml import ZestXML
# EURLEX = dict(shortyK=150, bs_count=120, bs_alpha=0.02, bs_direct_wt=0.8,
#               bilinear_classifier_cost=5, bilinear_normalize=0, device='auto', num_thread=0)

# m = ZestXML('GZXML-Datasets/GZ-Eurlex-4.3K', 'Results/Eurlex-ref', **EURLEX)
# m.fit(); m.predict()
# ref = m.evaluate()

In [ ]:
# SKIPPED — see benchmarks/RESULTS.md for what it measured.
# Uncomment this cell to re-run it.
# # does the pruning gain survive on a standard benchmark? min_sim 0.0 is the control above.
# # self-contained: fetch the vectors here rather than relying on a commented-out cell.
# import os
# VECTORS = '/content/glove100.gz'
# if not os.path.exists(VECTORS):
#     !wget -q -O {VECTORS} https://github.com/RaRe-Technologies/gensim-data/releases/download/glove-wiki-gigaword-100/glove-wiki-gigaword-100.gz

# print('%-10s %7s %7s %7s %9s %9s' % ('min_sim','P@1','P@5','PSP@5','unseen','seen'))
# print('%-10s %7.2f %7.2f %7.2f %9.2f %9.2f' % (0.0, ref['all labels']['P@1'],
#       ref['all labels']['P@5'], ref['all labels']['PSP@5'],
#       ref['unseen only']['P@1'], ref['seen only']['P@1']))
# for s in (0.15, 0.30, 0.45):
#     m = ZestXML('GZXML-Datasets/GZ-Eurlex-4.3K', f'Results/Eurlex-prune-{s}', **EURLEX,
#                 prune_vectors=VECTORS, prune_min_sim=s)
#     m.fit(); m.predict(); r = m.evaluate(verbose=False)
#     print('%-10s %7.2f %7.2f %7.2f %9.2f %9.2f' % (s, r['all labels']['P@1'],
#           r['all labels']['P@5'], r['all labels']['PSP@5'],
#           r['unseen only']['P@1'], r['seen only']['P@1']))

**Reading it.** The npm sweep was still climbing at 0.30, so 0.45 is included here to
find the turn. If pruning helps on EURLex too, it is a property of large mined patterns and
not of one self-built dataset — EURLex has 4,271 labels against npm's 3,223, so the pattern
budget should bind similarly.

If the control row does *not* match the paper, that is the more important finding and
everything measured in this repo needs re-reading. Note two caveats before concluding a
mismatch: the paper reports the C++ implementation, and this port differs from it
deliberately in three ways documented in the README (Adam on the primal objective rather
than dual coordinate descent, exact rather than approximate shortlisting, and consistent
normalisation between train and predict). The synthetic-data comparison put those at well
under a point, but they are not zero.

### 11b — before comparing against the paper, check the protocol

The reference `metrics.py` shipped with ZestXML does three things this repo's evaluator did
not, and any of them can move P@1 by several points:

1. **It defaults to `validation = True`**, scoring `val_X_Y` and not `tst_X_Y`.
2. **It applies a filter matrix** (`pos_trn_val.txt` / `pos_trn_tst.txt`) that zeroes
   (point, label) pairs before scoring. Skipping it inflates every metric.
3. **It combines four score matrices** — `clf`, `bilinear`, `knn`, `shortlist` — with
   weights `alpha/beta/gamma/delta` passed on the command line. A published number may come
   from a tuned combination, not from `score_alpha * bilinear + (1 - alpha) * knn`.

So run this before drawing any conclusion from the numbers above.

In [ ]:
# SKIPPED — see benchmarks/RESULTS.md for what it measured.
# Uncomment this cell to re-run it.
# import os
# D = 'GZXML-Datasets/GZ-Eurlex-4.3K'
# print('files shipped:')
# for f in sorted(os.listdir(D)):
#     print('   %-28s %10d bytes' % (f, os.path.getsize(f'{D}/{f}')))

# from zestxml.io import read_text_smat
# for split in ('trn', 'tst', 'val'):
#     p = f'{D}/{split}_X_Y.txt'
#     if os.path.exists(p):
#         m = read_text_smat(p)
#         print(f'{split}_X_Y: {m.nrows} points x {m.ncols} labels, {m.nnz} positives')

In [ ]:
# SKIPPED — see benchmarks/RESULTS.md for what it measured.
# Uncomment this cell to re-run it.
# # A score matrix only means something against the split it was PREDICTED on. Both splits
# # here have 6000 points, so a shape check cannot catch the mix-up -- scoring the test
# # predictions against val_X_Y silently gave P@1 4.20, which is what unrelated documents
# # look like. So: predict on val properly, then score it.
# from zestxml.eval import report

# print('=' * 70, '\nsplit = tst')
# report('Results/Eurlex-ref/score_mat.bin', D, split='tst')

# if os.path.exists(f'{D}/val_X_Xf.txt'):
#     print('=' * 70, '\nsplit = val (predicted on val_X_Xf, not reusing the test scores)')
#     v = ZestXML(D, 'Results/Eurlex-val', **EURLEX,
#                 tst_X_Xf=f'{D}/val_X_Xf.txt', tst_X_Y=f'{D}/val_X_Y.txt')
#     import shutil; os.makedirs('Results/Eurlex-val', exist_ok=True)
#     shutil.copytree('Results/Eurlex-ref/model', 'Results/Eurlex-val/model', dirs_exist_ok=True)
#     v.predict()
#     report('Results/Eurlex-val/score_mat.bin', D, split='val')
# else:
#     print('no val_X_Xf.txt -- the val split cannot be scored, only its labels ship')

**How to read it.** If a `pos_trn_*.txt` appears in the listing and the evaluator says
`no filter matrix found`, our numbers are unfiltered and inflated relative to the paper —
tell me the filename and I will wire it in. If `val_X_Y.txt` exists, the paper's headline
number is most likely the validation one, and the test row above is not the comparable
figure.

Independent of the protocol, one thing in the run above is worth noting: the `unseen only`
row covers **89 test points**, out of 6000. The zero-shot column on this dataset is a very
small sample, so treat single-digit differences there as noise regardless of what they are
being compared to.

## 12 — a benchmark with published numbers: LF-AmazonTitles-131K

Everything before this ran on datasets with no published counterpart, so "is 73.04 good?"
had no answer. This one does. **LF-AmazonTitles-131K** (294.8K train / 134.8K test / 131K
labels) is in essentially every recent XMC paper, and the encoder-only band is tight:

| method | P@1 | P@5 | PSP@5 |
|---|---|---|---|
| GraphFormers | 20.84 | 10.06 | 24.93 |
| SiameseXML | 41.42 | 21.21 | 46.19 |
| DPR | 41.85 | 20.88 | 49.45 |
| NGAME | 42.61 | 20.69 | 48.71 |
| ANCE | 42.67 | 20.98 | 49.03 |
| DEXML | 42.52 | 20.64 | 47.40 |
| DEXA | 44.76 | 21.18 | 49.50 |
| **PRIME** | **44.87** | **21.53** | **49.73** |

(PRIME, [arXiv:2410.20401](https://arxiv.org/abs/2410.20401), Table 2.)

**P@1 41–45 is the band.** ZestXML is a sparse linear model against transformer encoders,
so landing below it is expected — the useful question is *how far* below, because that
number is the honest price of its speed and its zero-shot ability. Landing anywhere near
20 would instead mean something is wrong with the conversion or the run.

Note this stage builds the **standard** split (`--unseen_frac 0`), not a zero-shot one. The
published numbers are on the standard benchmark, and matching a protocol matters more than
measuring the setting we happen to care about — that comes after.

In [ ]:
%cd /content/zestxml
!pip install -q gdown
import os, subprocess

# Drive and archive.org are both blocked from my sandbox, so none of these is verified by
# me -- each is tried in turn and the cell reports which one worked.
DATASET = 'amazoncat-13k'      # or 'amazontitles-131k' (the P@1 41-45 reference band)
RAW = f'raw/{DATASET}'
os.makedirs('raw', exist_ok=True)

DRIVE_IDS = {   # from github.com/Beneboe/Multi-Label-Text-Classification (trn.json/tst.json)
    'amazoncat-13k': '17rVRDarPwlMpb3l5zof9h34FlwbpTu4l',
}
CANDIDATES = []
if DATASET in DRIVE_IDS:
    CANDIDATES.append(('Google Drive (raw json)',
        f'cd raw && gdown -q --fuzzy "https://drive.google.com/file/d/{DRIVE_IDS[DATASET]}/view" '
        f'&& for f in *.zip; do unzip -q -o "$f" -d {DATASET}; done 2>/dev/null; '
        f'for f in *.tar.gz; do tar -xzf "$f"; done 2>/dev/null; true'))
CANDIDATES.append(('archive.org / PECOS xmc-base',
    f'wget -q --show-progress -O raw/{DATASET}.tar.gz '
    f'"https://archive.org/download/pecos-dataset/xmc-base/{DATASET}.tar.gz" '
    f'&& tar -xzf raw/{DATASET}.tar.gz -C raw && rm -f raw/{DATASET}.tar.gz'))

if not os.path.isdir(RAW) or not os.listdir(RAW):
    for name, cmd in CANDIDATES:
        print(f'trying {name} ...')
        subprocess.run(cmd, shell=True)
        if os.path.isdir(RAW) and os.listdir(RAW):
            print(f'  got it from {name}'); break
        print(f'  {name} failed')

print('\nraw/ now contains:')
for root, _, files in os.walk('raw'):
    for f in sorted(files)[:30]:
        print('   %-52s %12d bytes' % (os.path.join(root, f), os.path.getsize(os.path.join(root, f))))

if not (os.path.isdir(RAW) and os.listdir(RAW)):
    print("""
NOT DOWNLOADED. Get it by hand from
  http://manikvarma.org/downloads/XC/XMLRepository.html
unpack into raw/{}/ and rerun. make_xmc.py accepts:
  trn/tst/lbl.json[.gz]                                      (label text from lbl, or from
                                                              Yf.txt / output-items.txt /
                                                              label_map.txt beside it)
  X.{{trn,tst}}.txt + Y.{{trn,tst}}.npz + output-items.txt      (PECOS)
  {{train,test,label}}_raw_texts.txt + {{trn,tst}}_X_Y.txt
It needs LABEL TEXT -- a bundle with only trn/tst and no label names cannot be used, since
ZestXML scores a label through the words of its name.""".format(DATASET))

In [ ]:
!python benchmarks/datasets/make_xmc.py {RAW} GZXML-Datasets/{DATASET} --unseen_frac 0

In [ ]:
from zestxml import ZestXML
# 13K-131K labels is well beyond anything else here. max_elems / dense_elems cap the
# working set without changing the model -- lower them if this runs out of memory.
m = ZestXML(f'GZXML-Datasets/{DATASET}', f'Results/{DATASET}',
            shortyK=100, bs_count=40, bs_alpha=0.02, bs_direct_wt=0.8,
            bilinear_classifier_cost=5, bilinear_normalize=0,
            device='auto', num_thread=0, max_elems=1 << 23, dense_elems=1 << 23)
m.fit(); m.predict()
r = m.evaluate()
print('\nreference band, amazontitles-131k (PRIME Table 2): P@1 41-45, P@5 20.6-21.5, PSP@5 46-50')
print('this run: P@1 %.2f  P@5 %.2f  PSP@5 %.2f'
      % (r['all labels']['P@1'], r['all labels']['P@5'], r['all labels']['PSP@5']))

**How to read the outcome.**

* **P@1 in the mid-30s or higher** — the port is behaving sensibly for a sparse linear model
  on a 131K-label benchmark, and every relative comparison made earlier in this repo stands.
* **P@1 near 20 or below** — something is wrong in the conversion, not in the model. First
  suspects, in order: the label ids in `target_ind` not matching the row order of
  `lbl.json.gz`; documents built from `content` when the benchmark is title-only; `min_df`
  pruning too much of a short-title vocabulary.
* **A run that dies on memory** — lower `max_elems` / `dense_elems`. Neither changes the
  model, only how much is expanded at once.

Once this lands somewhere defensible, the zero-shot version is one flag away
(`--unseen_frac 0.1`), and *that* is the number worth comparing against the Zest-XML row in
the IRENE paper's Table 2.